# 13 · Claude Code CLI inside a MicroVM

How to run a full **agent CLI** (Anthropic's Claude Code)
*inside* a sandbox, with its model served by Bedrock.

**Key concepts**
- **Agent-in-a-sandbox** — instead of you calling an LLM, you run a complete coding
  agent (the `claude` CLI) inside the VM; it edits files and runs commands there,
  fully isolated from your machine.
- **Full-OS MicroVM** — needed because the CLI requires Node, package installs, and
  a real filesystem — only the `microvm` backend provides that (internet egress).
- **Claude Code on Bedrock** — set `CLAUDE_CODE_USE_BEDROCK=1` + `ANTHROPIC_MODEL`;
  the CLI authenticates with the VM's **execution role** (no API key inside the VM).

**How it works:** create the VM, install the CLI, pipe a prompt to `claude -p`, then
read back the files it generated.

**Prerequisites:** a MicroVM image with internet egress
(`RAITBOX_MICROVM_IMAGE`), an execution role with `bedrock:InvokeModel`
(`RAITBOX_EXECUTION_ROLE`), and Bedrock model access.

In [ ]:
%pip install -q raitbox

### Configuration & sandbox
Launch a MicroVM with Claude Code pointed at Bedrock via env vars (`CLAUDE_CODE_USE_BEDROCK=1`, a Bedrock model id) and the VM's execution role for auth.

In [ ]:
import os

from raitbox import Sandbox

IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]
EXEC_ROLE = os.environ["RAITBOX_EXECUTION_ROLE"]
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL = os.environ.get("ANTHROPIC_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
INTERNET_EGRESS = os.environ.get(
    "RAITBOX_EGRESS",
    f"arn:aws:lambda:{REGION}:aws:network-connector:aws-network-connector:INTERNET_EGRESS",
)

sbx = Sandbox.create(
    backend="microvm", template=IMAGE_ARN,
    envs={"CLAUDE_CODE_USE_BEDROCK": "1", "ANTHROPIC_MODEL": MODEL, "AWS_REGION": REGION,
          "IS_SANDBOX": "1"},  # allow --dangerously-skip-permissions as root in the VM
    execution_role_arn=EXEC_ROLE, egress_connectors=[INTERNET_EGRESS],
    region_name=REGION, timeout=1800,
)
print("Sandbox created", sbx.sandbox_id)

## Install Claude Code (E2B's template has it preinstalled)

In [ ]:
print(sbx.commands.run("dnf install -y nodejs npm >/dev/null 2>&1; node --version").stdout)
print(sbx.commands.run("npm install -g @anthropic-ai/claude-code >/dev/null 2>&1; claude --version").stdout)

## Run a prompt through Claude Code (backed by Bedrock)

In [ ]:
result = sbx.commands.run(
    "cd /tmp && echo 'Create a hello world index.html' | claude -p --dangerously-skip-permissions",
    timeout=600,
)
print(result.stdout)
print(sbx.files.read("/tmp/index.html"))
sbx.kill()

## Recap & where to go next
You ran a self-contained coding agent inside an isolated VM, model-backed by Bedrock.

**Related:** `12` (microvm background/network) · `14`/`15` (patch-based coding evals
in a microvm).